# Pandas Data Cleaning and Analysis

**AI Engineering School — Practical Data Preparation**

Real datasets usually contain missing values, inconsistent text, duplicate records, incorrect data types, and information spread across several tables.

In this notebook, you will learn how to:

- add, rename, and remove columns safely;
- detect and handle missing values;
- clean text and convert data types;
- work with dates;
- remove duplicates;
- aggregate data with `groupby()`;
- reshape data with pivot tables;
- combine tables with `merge()` and `concat()`;
- export cleaned data.

The examples use a small corpus-annotation dataset so that the techniques are directly relevant to NLP and AI projects.

## 1. Imports and display settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)

## 2. Create a deliberately messy dataset

In [ ]:
annotations = pd.DataFrame({
    "annotation_id": [1, 2, 3, 4, 5, 5, 6, 7],
    "text_id": ["T01", "T01", "T02", "T03", "T04", "T04", "T05", "T06"],
    "token": [" زبان ", "داده", "MODEL", " پیکره", "تحلیل", "تحلیل", None, "یادگیری ماشین"],
    "label": ["NOUN", "NOUN", "noun", "NOUN", "VERB", "VERB", "ADJ", "NOUN"],
    "confidence": [0.95, 0.88, np.nan, 0.91, 0.76, 0.76, 0.82, None],
    "annotator": ["A", "A", "B", "B", "A", "A", None, "B"],
    "annotation_date": [
        "2026-01-05", "2026-01-05", "2026/01/06", "2026-01-07",
        "not recorded", "not recorded", "2026-01-09", "2026-01-10"
    ]
})

annotations

In [ ]:
annotations.info()

## 3. Work on a copy

When experimenting, make an explicit copy so that you can always return to the original data.

In [ ]:
clean = annotations.copy()

## 4. Rename, add, and remove columns

Use clear, consistent column names—usually lowercase words separated by underscores.

In [ ]:
clean = clean.rename(columns={
    "label": "pos_tag",
    "annotation_date": "date"
})

clean.head()

### Add calculated columns

Vectorized expressions are normally faster, shorter, and clearer than Python loops.

In [ ]:
clean["is_high_confidence"] = clean["confidence"] >= 0.90
clean["token_length"] = clean["token"].str.len()

clean.head()

Because the original token strings contain extra spaces, their current lengths are not yet reliable. We will clean the text shortly.

### Remove columns

Prefer assigning the returned DataFrame. This makes the transformation visible in your code.

In [ ]:
clean = clean.drop(columns=["is_high_confidence"])
clean.head()

## 5. Detect missing values

In [ ]:
clean.isna()

In [ ]:
clean.isna().sum().sort_values(ascending=False)

In [ ]:
missing_percentage = clean.isna().mean().mul(100).round(1)
missing_percentage.sort_values(ascending=False)

Missing values are not automatically “bad.” The correct treatment depends on why the values are absent and how the column will be used.

## 6. Handle missing values

### Fill categorical values

A transparent placeholder such as `"Unknown"` is often better than silently guessing a category.

In [ ]:
clean["annotator"] = clean["annotator"].fillna("Unknown")

### Fill numeric values

For a small demonstration, we fill missing confidence values with the median. In a real project, document this decision and consider whether imputation is scientifically justified.

In [ ]:
confidence_median = clean["confidence"].median()
clean["confidence"] = clean["confidence"].fillna(confidence_median)

print("Median used:", confidence_median)

### Missing text

A missing token cannot be meaningfully analyzed, so we remove rows where `token` is absent.

In [ ]:
clean = clean.dropna(subset=["token"]).copy()

In [ ]:
clean.isna().sum()

A common mistake is calling `fillna()` without saving its result:

```python
df["column"].fillna(value)  # displayed, but df is not changed
```

Use assignment instead:

```python
df["column"] = df["column"].fillna(value)
```

## 7. Clean text with `.str`

In [ ]:
clean["token"] = (
    clean["token"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

clean["pos_tag"] = clean["pos_tag"].str.strip().str.upper()
clean["token_length"] = clean["token"].str.len()

clean[["token", "pos_tag", "token_length"]]

Useful string methods include:

- `.str.lower()` and `.str.upper()`;
- `.str.strip()` for surrounding whitespace;
- `.str.replace()` for replacements;
- `.str.contains()` for matching;
- `.str.split()` for tokenizing simple delimiter-separated strings.

For Persian NLP, full normalization often requires a dedicated normalizer in addition to basic Pandas string methods.

In [ ]:
clean.loc[
    clean["token"].str.contains("یادگیری", na=False),
    ["text_id", "token", "pos_tag"]
]

## 8. Remove duplicate rows

First inspect duplicates instead of deleting them blindly.

In [ ]:
clean.loc[clean.duplicated(keep=False)]

Here, the repeated row has the same `annotation_id` and the same content, so we keep the first copy.

In [ ]:
clean = clean.drop_duplicates().copy()
clean

When only certain columns define a unique record, use `subset`:

```python
df.drop_duplicates(subset=["text_id", "token", "annotator"])
```

## 9. Convert data types

Correct data types prevent subtle errors and make analysis more efficient.

In [ ]:
clean["annotation_id"] = clean["annotation_id"].astype("int64")
clean["pos_tag"] = clean["pos_tag"].astype("category")
clean["annotator"] = clean["annotator"].astype("category")

clean.dtypes

### Convert dates safely

`errors="coerce"` converts invalid date strings to `NaT`, Pandas' missing datetime value.

In [ ]:
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
clean[["date"]]

In [ ]:
clean["year"] = clean["date"].dt.year
clean["month"] = clean["date"].dt.month
clean["day_name"] = clean["date"].dt.day_name()

clean[["date", "year", "month", "day_name"]]

## 10. Conditional columns with `np.where()` and `.map()`

In [ ]:
clean["confidence_level"] = np.where(
    clean["confidence"] >= 0.90,
    "high",
    "needs_review"
)

clean[["confidence", "confidence_level"]]

Use `.map()` for simple value-to-value mappings.

In [ ]:
tag_descriptions = {
    "NOUN": "noun",
    "VERB": "verb",
    "ADJ": "adjective"
}

clean["tag_description"] = clean["pos_tag"].map(tag_descriptions)
clean[["pos_tag", "tag_description"]]

## 11. Grouping and aggregation

`groupby()` follows the **split–apply–combine** pattern:

1. split rows into groups;
2. apply a calculation to each group;
3. combine the results.

In [ ]:
clean.groupby("pos_tag", observed=True)["confidence"].mean()

In [ ]:
tag_summary = (
    clean.groupby("pos_tag", observed=True)
    .agg(
        annotation_count=("annotation_id", "count"),
        mean_confidence=("confidence", "mean"),
        unique_texts=("text_id", "nunique"),
        average_token_length=("token_length", "mean")
    )
    .sort_values("annotation_count", ascending=False)
)

tag_summary.round(2)

Named aggregation makes the output column names explicit and readable.

In [ ]:
annotator_summary = (
    clean.groupby("annotator", observed=True)
    .agg(
        annotations=("annotation_id", "count"),
        average_confidence=("confidence", "mean")
    )
    .round(3)
)

annotator_summary

## 12. Pivot tables and cross-tabulations

In [ ]:
pd.crosstab(
    clean["annotator"],
    clean["pos_tag"],
    margins=True
)

In [ ]:
pd.pivot_table(
    clean,
    index="annotator",
    columns="pos_tag",
    values="confidence",
    aggfunc="mean",
    observed=True
).round(3)

Use `crosstab()` mainly for counts and `pivot_table()` for flexible numerical summaries.

## 13. Combine tables with `merge()`

A second table may contain document-level metadata.

In [ ]:
documents = pd.DataFrame({
    "text_id": ["T01", "T02", "T03", "T04", "T05", "T06"],
    "genre": ["news", "academic", "fiction", "news", "social_media", "academic"],
    "language": ["Persian", "English", "Persian", "Persian", "Persian", "Persian"]
})

documents

In [ ]:
enriched = clean.merge(
    documents,
    on="text_id",
    how="left",
    validate="many_to_one"
)

enriched

The `validate` argument checks your assumption about the relationship between tables. Here, many annotation rows may belong to one document.

Common join types:

- `left`: keep every row from the left table;
- `inner`: keep only matching rows;
- `outer`: keep all rows from both tables;
- `right`: keep every row from the right table.

In [ ]:
enriched.groupby("genre")["confidence"].mean().sort_values(ascending=False)

## 14. Stack tables with `concat()`

Use `concat()` when tables have the same columns and represent additional rows.

In [ ]:
new_annotations = pd.DataFrame({
    "annotation_id": [8, 9],
    "text_id": ["T07", "T07"],
    "token": ["هوش مصنوعی", "کاربردی"],
    "pos_tag": ["NOUN", "ADJ"],
    "confidence": [0.93, 0.89],
    "annotator": ["A", "A"],
    "date": pd.to_datetime(["2026-01-11", "2026-01-11"]),
    "token_length": [10, 7],
    "year": [2026, 2026],
    "month": [1, 1],
    "day_name": ["Sunday", "Sunday"],
    "confidence_level": ["high", "needs_review"],
    "tag_description": ["noun", "adjective"]
})

combined = pd.concat([clean, new_annotations], ignore_index=True)
combined.tail()

## 15. Vectorization, `.map()`, and `.apply()`

Prefer built-in vectorized methods whenever possible.

```python
df["revenue"] = df["units"] * df["price"]       # vectorized
df["label_name"] = df["label"].map(mapping)     # simple mapping
```

Use `.apply()` for logic that cannot be expressed clearly with vectorized operations, but avoid using it as the first solution.

In [ ]:
def review_message(confidence):
    if confidence >= 0.90:
        return "Ready"
    return "Manual review"

clean["review_status"] = clean["confidence"].apply(review_message)
clean[["confidence", "review_status"]]

The previous example is educational. The equivalent `np.where()` solution is usually faster for large datasets.

## 16. Basic plotting with Pandas

Pandas plotting methods use Matplotlib.

In [ ]:
tag_counts = clean["pos_tag"].value_counts()

ax = tag_counts.plot(
    kind="bar",
    title="Number of annotations by POS tag",
    xlabel="POS tag",
    ylabel="Count"
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 17. Export cleaned data

For CSV files containing Persian text, `utf-8-sig` is often convenient when the file will be opened in Microsoft Excel.

```python
clean.to_csv(
    "clean_annotations.csv",
    index=False,
    encoding="utf-8-sig"
)
```

Other useful formats:

```python
clean.to_json("clean_annotations.json", orient="records", force_ascii=False)
clean.to_excel("clean_annotations.xlsx", index=False)
clean.to_parquet("clean_annotations.parquet", index=False)
```

Parquet usually preserves data types better and is more space-efficient than CSV, but it may require an additional engine such as `pyarrow`.

## 18. A reusable cleaning pipeline

Chaining transformations can make a workflow easier to reproduce.

In [ ]:
def prepare_annotations(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {
        "annotation_id", "text_id", "token",
        "label", "confidence", "annotator", "annotation_date"
    }
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    result = (
        df.rename(columns={
            "label": "pos_tag",
            "annotation_date": "date"
        })
        .dropna(subset=["token"])
        .drop_duplicates()
        .copy()
    )

    result["token"] = (
        result["token"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    result["pos_tag"] = result["pos_tag"].str.strip().str.upper()
    result["confidence"] = result["confidence"].fillna(
        result["confidence"].median()
    )
    result["annotator"] = result["annotator"].fillna("Unknown")
    result["date"] = pd.to_datetime(result["date"], errors="coerce")

    return result.reset_index(drop=True)

In [ ]:
prepared_again = prepare_annotations(annotations)
prepared_again

## 19. Final quality checks

Before modeling or publishing a dataset, verify:

In [ ]:
print("Rows and columns:", clean.shape)
print("\nMissing values:")
print(clean.isna().sum())
print("\nDuplicate rows:", clean.duplicated().sum())
print("\nData types:")
print(clean.dtypes)

Additional checks may include:

- valid label sets;
- allowed numerical ranges;
- uniqueness of identifiers;
- agreement between annotators;
- class balance;
- train/validation/test leakage;
- encoding and normalization of Persian characters;
- documentation of every cleaning decision.

## Key takeaways

- Never clean data blindly; inspect the problem and document each decision.
- Save transformation results or use explicit method chaining.
- Use vectorized operations before loops or `.apply()`.
- Convert dates and categories to suitable data types.
- Use `groupby()`, pivot tables, and merges to move from raw rows to useful analysis.
- Validate joins and run final quality checks before machine learning.